[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/S10_03_regularizacion.ipynb)

# S10 · Práctica 3 — Entrenar bien: sobreajuste, regularización y callbacks

**Tiempo estimado:** 1 h 45 min · **Sprint 10** (UD 4.3)

**Contexto TurisData.** Una red que brilla en entrenamiento y falla con reservas nuevas es peor que inútil:
da una falsa seguridad al cliente. Aquí aprenderás a **detectar** el sobreajuste con curvas de aprendizaje y a
**controlarlo** con las herramientas habituales.

**Al terminar sabrás:**
1. Diagnosticar sobreajuste leyendo curvas de entrenamiento y validación.
2. Aplicar regularización L2, *dropout*, parada temprana (*early stopping*) y *batch normalization*.
3. Ver el efecto de la tasa de aprendizaje y del tamaño de lote.
4. Usar *callbacks* (parada, reducción de tasa, punto de guardado) y guardar/cargar un modelo.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, tempfile
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 0. Datos: el mismo preprocesado de la práctica anterior
Para ver bien el sobreajuste usaremos **solo 800 reservas de entrenamiento** (con pocos datos y una red grande,
memorizar es facilísimo). Validación: 960 reservas. La prueba final la reservamos para el reto.

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
num = ["antelacion_dias", "noches", "adultos", "ninos", "peticiones_especiales", "cliente_repetidor",
       "cancelaciones_previas", "mes_llegada", "precio_noche"]
cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[num + cat], df["cancelada"].to_numpy()
X_res, _, y_res, _ = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)   # se descarta la prueba
X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=SEMILLA, stratify=y_res)
pre = ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
A_ent = pre.fit_transform(X_ent).astype("float32")
A_val = pre.transform(X_val).astype("float32")

A_peq, y_peq = A_ent[:800], y_ent[:800]     # entrenamiento pequeño para provocar sobreajuste
print("Entrenamiento pequeño:", A_peq.shape, "· Entrenamiento completo:", A_ent.shape, "· Validación:", A_val.shape)

## 1. Provocar y diagnosticar el sobreajuste

**Idea.** Una red con muchísimos parámetros para pocos datos acaba **aprendiéndose de memoria** los ejemplos,
incluido el ruido. Se reconoce porque:
- la pérdida de **entrenamiento** sigue bajando,
- la pérdida de **validación** baja al principio, toca un mínimo y **vuelve a subir**.

La distancia entre ambas curvas es la "brecha de generalización".

In [ ]:
def construir(n_entradas, capas=(256, 128), l2=0.0, dropout=0.0, batchnorm=False, tasa=1e-3):
    """MLP de clasificación con regularización opcional."""
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    m = keras.Sequential([keras.layers.Input(shape=(n_entradas,))])
    for u in capas:
        m.add(keras.layers.Dense(u, activation="relu", kernel_regularizer=reg, use_bias=not batchnorm))
        if batchnorm:
            m.add(keras.layers.BatchNormalization())
            m.add(keras.layers.Activation("relu"))   # (se vuelve a aplicar ReLU tras normalizar)
        if dropout > 0:
            m.add(keras.layers.Dropout(dropout))
    m.add(keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=keras.optimizers.Adam(tasa), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc")])
    return m

def entrenar(A, y_, epocas=60, lote=32, callbacks=None, **cfg):
    keras.utils.set_random_seed(SEMILLA)
    m = construir(A.shape[1], **cfg)
    h = m.fit(A, y_, validation_data=(A_val, y_val), epochs=epocas, batch_size=lote,
              callbacks=callbacks, verbose=0)
    return m, h

def curvas(h, titulo, ax=None):
    ax = ax or plt.gca()
    ax.plot(h.history["loss"], label="entrenamiento")
    ax.plot(h.history["val_loss"], label="validación")
    ax.set_xlabel("época"); ax.set_ylabel("pérdida"); ax.set_title(titulo); ax.legend()

In [ ]:
m_base, h_base = entrenar(A_peq, y_peq, epocas=60)
plt.figure(figsize=(6, 4)); curvas(h_base, "Red grande y pocos datos: sobreajuste"); plt.show()

**Ejercicio 1.** A partir de `h_base.history["val_loss"]` calcula:
- `epoca_optima`: la época (contando desde 1) en la que la pérdida de validación fue **mínima**;
- `brecha_final`: pérdida de validación de la última época **menos** pérdida de entrenamiento de la última época.

Si `epoca_optima` es muy inferior al total de épocas, seguir entrenando solo empeoraba el modelo.

In [ ]:
# TODO: np.argmin(val_loss) + 1; y la resta de los últimos valores de val_loss y loss
...

assert 1 <= epoca_optima <= 60, "epoca_optima debe estar entre 1 y 60 (recuerda sumar 1 porque Python cuenta desde 0)"
assert brecha_final > 0.2, "Debería haber una brecha clara (validación bastante peor que entrenamiento). Revisa el signo de la resta"
print(f"Mejor época: {epoca_optima} de 60 · brecha final = {brecha_final:.2f}")

## 2. Las herramientas de regularización

| Técnica | Idea en una frase | Cómo se pone en Keras |
|---|---|---|
| **L2** (*weight decay*) | Castiga los pesos grandes: la red prefiere soluciones "suaves" | `kernel_regularizer=keras.regularizers.l2(0.01)` |
| **Dropout** | En cada paso "apaga" al azar un % de neuronas: no puede depender de una sola | `keras.layers.Dropout(0.3)` |
| **Early stopping** | Parar cuando la validación deja de mejorar | callback `EarlyStopping` |
| **Batch normalization** | Reescala las activaciones en cada lote: entrena más estable y rápido; regulariza algo | `keras.layers.BatchNormalization()` |

La función `construir` de arriba ya admite `l2`, `dropout` y `batchnorm`. Vamos a probar cada una **por separado**.

**Ejercicio 2.** Entrena con `entrenar(A_peq, y_peq, ...)` una red con **dropout de 0.4** y guarda el modelo en
`m_drop` y el historial en `h_drop`. Después comprueba en `n_dropout` cuántas capas `Dropout` tiene el modelo
(usa `isinstance(capa, keras.layers.Dropout)`).

In [ ]:
# TODO: entrenar(A_peq, y_peq, epocas=60, dropout=0.4) y cuenta las capas Dropout de m_drop.layers
...

assert n_dropout == 2, "Con capas=(256, 128) debe haber 2 capas Dropout (una tras cada capa oculta)"
assert len(h_drop.history["loss"]) == 60, "Entrena 60 épocas"

In [ ]:
configs = {
    "Sin regularizar": dict(),
    "L2 = 0.01": dict(l2=0.01),
    "Dropout 0.4": dict(dropout=0.4),
    "BatchNorm": dict(batchnorm=True),
    "L2 + Dropout": dict(l2=0.005, dropout=0.3),
}
resultados = {}
fig, ejes = plt.subplots(1, len(configs), figsize=(4 * len(configs), 3.6), sharey=True)
for eje, (nombre, cfg) in zip(ejes, configs.items()):
    m, h = entrenar(A_peq, y_peq, epocas=60, **cfg)
    resultados[nombre] = h.history
    curvas(h, nombre, eje)
plt.tight_layout(); plt.show()

resumen = pd.DataFrame({
    nombre: {"mejor pérdida val.": min(h["val_loss"]), "pérdida val. final": h["val_loss"][-1],
             "AUC val. final": h["val_auc"][-1], "AUC val. máximo": max(h["val_auc"])}
    for nombre, h in resultados.items()
}).T.round(3)
resumen

**Cómo leerlo.** Fíjate en la columna *pérdida val. final* frente a *mejor pérdida val.*: cuanto más se parecen,
menos sobreajuste. La regularización rara vez "dispara" la métrica; lo que hace es **evitar que empeore** y hacer
el modelo más fiable. Con tan pocos datos, muchas veces el mayor logro es simplemente no hacer el ridículo.

## 3. Parada temprana y callbacks

**Idea.** Un *callback* es una función que Keras ejecuta al final de cada época (o lote). Los tres imprescindibles:
- `EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)`: para si la validación no mejora
  en 8 épocas seguidas y **devuelve los pesos de la mejor época**.
- `ReduceLROnPlateau(factor=0.5, patience=4)`: si se estanca, **baja la tasa de aprendizaje** a la mitad.
- `ModelCheckpoint(ruta, save_best_only=True)`: guarda el mejor modelo en disco.

(Existe también `TensorBoard`, un panel web para ver curvas; en Colab se activa con `%load_ext tensorboard`. Aquí basta con `matplotlib`.)

**Ejercicio 3.** Crea la lista `mis_callbacks` con un `EarlyStopping` (monitoriza `val_loss`, `patience=8`,
`restore_best_weights=True`) y un `ReduceLROnPlateau` (monitoriza `val_loss`, `factor=0.5`, `patience=4`).
Después entrena con `entrenar(A_peq, y_peq, epocas=100, callbacks=mis_callbacks, l2=0.005, dropout=0.3)`
y guarda el resultado en `m_es` y `h_es`.

In [ ]:
# TODO: keras.callbacks.EarlyStopping(...) y keras.callbacks.ReduceLROnPlateau(...) dentro de una lista
...

epocas_reales = len(h_es.history["loss"])
assert any(isinstance(c, keras.callbacks.EarlyStopping) for c in mis_callbacks), "Falta EarlyStopping"
assert any(isinstance(c, keras.callbacks.ReduceLROnPlateau) for c in mis_callbacks), "Falta ReduceLROnPlateau"
assert epocas_reales < 100, "La parada temprana debería haber cortado antes de las 100 épocas"
print(f"Entrenó {epocas_reales} de 100 épocas posibles. Mejor pérdida de validación: {min(h_es.history['val_loss']):.3f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
curvas(h_es, "Con EarlyStopping y ReduceLROnPlateau", ejes[0])
if "learning_rate" in h_es.history:
    ejes[1].plot(h_es.history["learning_rate"]); ejes[1].set_yscale("log")
    ejes[1].set_xlabel("época"); ejes[1].set_ylabel("tasa de aprendizaje"); ejes[1].set_title("Cómo bajó la tasa")
plt.tight_layout(); plt.show()

## 4. Tasa de aprendizaje y tamaño de lote

Ahora con **todos** los datos de entrenamiento (3840 filas):
- **Tasa de aprendizaje** = tamaño del paso. Muy alta: la pérdida salta y no converge. Muy baja: tarda una eternidad.
- **Tamaño de lote** = cuántas filas por actualización. Lotes pequeños: más actualizaciones, más ruido (a veces
  regulariza). Lotes grandes: más estables y rápidos por época, pero con menos actualizaciones.

In [ ]:
import time
filas = []
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
for tasa in (1e-1, 1e-2, 1e-3, 1e-4):
    t0 = time.time()
    _, h = entrenar(A_ent, y_ent, epocas=15, lote=64, capas=(64, 32), tasa=tasa)
    ejes[0].plot(h.history["val_loss"], label=f"tasa {tasa:g}")
    filas.append({"parámetro": f"tasa {tasa:g}", "AUC val. final": h.history["val_auc"][-1], "segundos": time.time() - t0})
ejes[0].set_xlabel("época"); ejes[0].set_ylabel("pérdida de validación"); ejes[0].set_title("Efecto de la tasa de aprendizaje"); ejes[0].legend()
for lote in (32, 128, 512):
    t0 = time.time()
    _, h = entrenar(A_ent, y_ent, epocas=15, lote=lote, capas=(64, 32))
    ejes[1].plot(h.history["val_loss"], label=f"lote {lote}")
    filas.append({"parámetro": f"lote {lote}", "AUC val. final": h.history["val_auc"][-1], "segundos": time.time() - t0})
ejes[1].set_xlabel("época"); ejes[1].set_ylabel("pérdida de validación"); ejes[1].set_title("Efecto del tamaño de lote"); ejes[1].legend()
plt.tight_layout(); plt.show()
pd.DataFrame(filas).round(3)

**Qué observar en tus gráficos:** con una tasa alta (0.1 o 0.01) la pérdida de validación toca su mínimo enseguida y
luego se dispara (la red memoriza más rápido); con una tasa baja (0.001, 0.0001) la curva es más suave y tarda más
en sobreajustar. Con lotes grandes cada época cuesta menos segundos pero hay menos actualizaciones. No hay un valor
"correcto" universal: se elige mirando curvas y tiempo.

## 5. Guardar y cargar un modelo

Un modelo entrenado solo sirve si se puede **reutilizar sin volver a entrenar**. El formato actual es `.keras`
(guarda arquitectura + pesos + configuración del optimizador). **Ojo:** al cargarlo en producción hay que aplicar
exactamente el mismo preprocesado (`pre`); lo ideal es guardarlo también (por ejemplo con `joblib`).

**Ejercicio 4.** Guarda `m_es` en el archivo indicado por `ruta_modelo` con `m_es.save(...)`, cárgalo con
`keras.models.load_model(...)` en `m_cargado` y calcula en `prob_cargado` sus probabilidades sobre `A_val`.
Se comprobará que son idénticas a las del modelo original.

In [ ]:
carpeta_tmp = tempfile.mkdtemp()
ruta_modelo = os.path.join(carpeta_tmp, "modelo_cancelaciones.keras")
# TODO: m_es.save(ruta_modelo); m_cargado = keras.models.load_model(ruta_modelo); .predict(A_val)
...

prob_original = m_es.predict(A_val, verbose=0)
assert os.path.exists(ruta_modelo), "No se encuentra el archivo: ¿has llamado a m_es.save(ruta_modelo)?"
assert np.allclose(prob_original, prob_cargado, atol=1e-6), "El modelo cargado debe dar las mismas predicciones que el original"
print("Modelo guardado y recargado correctamente:", round(os.path.getsize(ruta_modelo) / 1024), "KB")

In [ ]:
# Limpieza del archivo temporal (en tu proyecto real, conserva el modelo)
import shutil
shutil.rmtree(carpeta_tmp, ignore_errors=True)

## 6. Mini-reto integrador: gana la red

Con el **entrenamiento completo** (`A_ent`, 3840 filas) parte de la red grande sin regularizar
(`capas=(256, 128)`, 40 épocas, lote 64) y consigue una **menor pérdida de validación del modelo final** combinando lo aprendido:
guarda en `m_final` tu mejor modelo (con `entrenar`) y en `mejor_perdida_val` su pérdida sobre `A_val`
(`m_final.evaluate(A_val, y_val, verbose=0)[0]`). Tienes que superar (tener menos pérdida que) la red grande
entrenada a ciegas, `perdida_referencia`.
Anota qué combinación usaste y por qué.

In [ ]:
_, h_ref = entrenar(A_ent, y_ent, epocas=40, lote=64, capas=(256, 128))
perdida_referencia = h_ref.history["val_loss"][-1]     # lo que obtendrías entrenando "a ciegas" hasta el final
print("Referencia (sin regularizar), pérdida de validación tras 40 épocas:", round(perdida_referencia, 4),
      "· la mejor había sido", round(min(h_ref.history["val_loss"]), 4))

In [ ]:
# TODO: Prueba dropout/L2 y callbacks: entrenar(A_ent, y_ent, epocas=40, lote=64, callbacks=..., ...)
...

assert mejor_perdida_val < perdida_referencia, "Aún no supera a la referencia: prueba una red más pequeña, dropout o L2"
print(f"Tu mejor pérdida de validación: {mejor_perdida_val:.4f} (referencia {perdida_referencia:.4f})")

**Tu combinación y por qué (edita esta celda):** …

## 7. Preguntas de reflexión
1. La pérdida de entrenamiento baja a 0,1 y la de validación sube a 1,2. ¿Qué diagnóstico y qué tres medidas tomarías?
2. ¿Por qué *dropout* solo actúa durante el entrenamiento y no cuando se predice?
3. ¿Por qué `restore_best_weights=True` importa en `EarlyStopping`?
4. Si tuvieras 10 veces más reservas, ¿el sobreajuste sería mayor o menor? ¿Por qué?

## 8. Resumen: qué has aprendido
- Sobreajuste = entrenamiento mejora, validación empeora. Se diagnostica con curvas, no con una cifra suelta.
- L2, dropout, parada temprana y *batch norm* son herramientas para reducir la brecha; el remedio más barato suele ser más datos o un modelo más pequeño.
- Callbacks: parar a tiempo, bajar la tasa al estancarse y guardar lo mejor.
- Tasa de aprendizaje y tamaño de lote se afinan mirando curvas.
- Se guarda con `.save("x.keras")` y se recupera con `load_model`, sin olvidar el preprocesado.